# Detectar caixes amb OpenCV, sense entrenar una xarxa

Aquest notebook construeix un banc de plantilles **només de train** i mesura el detector clàssic sobre les famílies de validació. No necessita GPU, PyTorch ni OCR. Les referències tenen anotacions; les imatges de validació només les utilitzen després per calcular les mètriques.

Puja `artifacts/colab_bundle.zip` a **My Drive/subsurface-bboxes/**. Si ja tens el projecte i el dataset a Colab, també pots extreure `opencv_update.zip` al projecte existent i ajustar `WORKDIR` i `DATASET`, sense tornar a pujar les imatges.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/subsurface-bboxes')
BUNDLE_PATH = DRIVE_ROOT / 'colab_bundle.zip'
WORKDIR = Path('/content/subsurface-opencv')
RUN_DIR = DRIVE_ROOT / 'opencv-runs' / datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
print('Resultats:', RUN_DIR)


In [ ]:
import zipfile
import subprocess
import sys
import os

if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f'No s’ha trobat {BUNDLE_PATH}')
WORKDIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as bundle:
    for member in bundle.infolist():
        if not (WORKDIR / member.filename).resolve().is_relative_to(WORKDIR.resolve()):
            raise ValueError(f'Ruta invàlida al ZIP: {member.filename}')
    bundle.extractall(WORKDIR)
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-cv.txt'], check=True)
DATASET = WORKDIR / 'dataset'
AUDIT = WORKDIR / 'artifacts' / 'cv_dataset'
TEMPLATES = RUN_DIR / 'templates'


## Separar les famílies i construir les referències

L’auditoria no descodifica els píxels ni exporta retalls. Després OpenCV extreu característiques ORB de la capçalera i el peu de les imatges d’entrenament. Una alineació independent del peu permet localitzar el final encara que l’altura total sigui diferent.


In [ ]:
import json
from bbox_detector.data import prepare_dataset
from bbox_detector.classical import build_template_bank

manifest = prepare_dataset(DATASET, AUDIT, materialize=False)
print(json.dumps(manifest['summary'], indent=2))
if (TEMPLATES / 'bank.json').is_file():
    bank = json.loads((TEMPLATES / 'bank.json').read_text())
else:
    bank = build_template_bank(AUDIT / 'manifest.json', TEMPLATES)
print('Referències de train:', len(bank['references']))


## Benchmark clàssic sobre documents nous

Es comparen les caixes finals amb les anotacions, conservant tots els errors i absències amb IoU zero. El detector pot rebutjar un format que no reconegui: és una limitació de cobertura que apareixerà a l’informe. El codi comprova que les referències no incloguin imatges o famílies reservades.


In [ ]:
from bbox_detector.benchmark import benchmark_manifest

benchmark = benchmark_manifest(
    AUDIT / 'manifest.json', RUN_DIR / 'classical_benchmark',
    templates=TEMPLATES, save_previews=True,
)
print(json.dumps(benchmark['backends'], indent=2))
print('Informe:', RUN_DIR / 'classical_benchmark' / 'benchmark.json')


## Predir un TIFF sense proporcionar-ne anotació

Canvia `IMAGE_PATH` per una imatge nova. Un error explícit indica que el banc no proporciona una alineació prou fiable; el notebook continua perquè puguis exportar les referències i els diagnòstics.


In [ ]:
from bbox_detector.classical import ClassicalDetector
from bbox_detector.inference import InferenceError
from bbox_detector.common import write_json

IMAGE_PATH = Path(next(r['image_path'] for r in manifest['records'] if r['split'] == 'val'))
diagnostics = {}
try:
    prediction = ClassicalDetector(TEMPLATES).predict_document(IMAGE_PATH, diagnostics=diagnostics)
except InferenceError as error:
    print(error)
    write_json(RUN_DIR / 'example_error.json', {'image_name': IMAGE_PATH.name, 'error': str(error), 'partial_boxes': error.partial_boxes, 'diagnostics': error.diagnostics})
else:
    write_json(RUN_DIR / 'example_prediction.json', prediction)
    print(json.dumps(prediction, indent=2))


## Opcional: comparar YOLO i l’híbrid amb els teus pesos

Aquesta part necessita les dependències YOLO i un `best.pt` entrenat amb les tres classes. Deixa `YOLO_WEIGHTS = None` per completar només OpenCV. Si estàs en CPU, la inferència de YOLO pot ser més lenta. L’híbrid proposa ajustos conservadors de les vores X; el benchmark indica quins documents milloren o empitjoren.


In [ ]:
# Exemple: Path('/content/drive/MyDrive/subsurface-bboxes/runs/.../training/detector/weights/best.pt')
YOLO_WEIGHTS = None

if YOLO_WEIGHTS is not None:
    YOLO_WEIGHTS = Path(YOLO_WEIGHTS)
    if not YOLO_WEIGHTS.is_file():
        raise FileNotFoundError(YOLO_WEIGHTS)
    # Run this optional block in the existing YOLO environment.
    # Avoid mixing OpenCV GUI/headless packages in a live CPU runtime.
    import importlib.util
    if importlib.util.find_spec('ultralytics') is None:
        raise RuntimeError('Per comparar YOLO, executa aquest bloc al notebook d’entrenament amb el codi actualitzat i la ruta TEMPLATES d’aquest banc.')
    import torch
    selected_device = '0' if torch.cuda.is_available() else 'cpu'
    comparison = benchmark_manifest(
        AUDIT / 'manifest.json', RUN_DIR / 'three_way_benchmark',
        templates=TEMPLATES, weights=YOLO_WEIGHTS, device=selected_device,
    )
    print(json.dumps(comparison['backends'], indent=2))
    print(json.dumps(comparison['paired_yolo_hybrid'], indent=2))
else:
    print('Comparació YOLO/híbrid omesa: no s’han proporcionat pesos entrenats.')


## Exportar el detector clàssic portable

El paquet exportat inclou el codi i les característiques de referència, sense necessitar els TIFF o JSON originals per predir. La precisió i cobertura del banc són les mesurades al benchmark, no les del mateix document de referència.


In [ ]:
import shutil

EXPORT = RUN_DIR / 'classical_export'
EXPORT.mkdir(parents=True, exist_ok=True)
for name in ['predict_bboxes.py', 'evaluate_bboxes.py', 'requirements-cv.txt', 'README.md']:
    shutil.copy2(WORKDIR / name, EXPORT / name)
for directory in ['tools', 'docs']:
    shutil.copytree(WORKDIR / directory, EXPORT / directory, dirs_exist_ok=True)
shutil.copytree(WORKDIR / 'bbox_detector', EXPORT / 'bbox_detector', dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__'))
shutil.copytree(TEMPLATES, EXPORT / 'templates', dirs_exist_ok=True)
archive = shutil.make_archive(str(RUN_DIR / 'classical_export'), 'zip', EXPORT)
print('Paquet portable:', archive)
print('Comanda: python predict_bboxes.py --backend classical --templates templates --image document.tif --output document.json')
